In [302]:
import duckdb as ddb

# rel_program_beslut = ddb.read_csv("data/resultat-ansokningsomgang-2020-2024-beslut.csv")
# rel_program_beslut.to_parquet("data/resultat-ansokningsomgang-2020-2024-beslut.parquet")
rel_program_beslut = ddb.read_parquet("data/resultat-ansokningsomgang-2020-2024-beslut.parquet")

# rel_program_diarie_kommun = ddb.read_csv("data/resultat-ansokningsomgang-2020-2024-diarie_kommun.csv")
# rel_program_diarie_kommun.to_parquet("data/resultat-ansokningsomgang-2020-2024-diarie_kommun.parquet")
rel_program_diarie_kommun = ddb.read_parquet("data/resultat-ansokningsomgang-2020-2024-diarie_kommun.parquet")


In [303]:
rel_diarie_kommun = ddb.sql(
    """
    select
        Diarienummer,
        Län,
        Kommun,
        -- Beslut
    from rel_program_diarie_kommun
    where Beslut = true
    """
)

In [304]:
rel_flera_kommuner = ddb.sql(
    """
    select
        pb."Ansökningsomgång",
        pb."Diarienummer",
        pb."Beslut",
        pb."Utbildningsområde",
        pb."Utbildningsnamn",
        dk."Län",                -- 'Flera kommuner'
        dk."Kommun",             -- 'Flera kommuner'
        pb."Antal kommuner",
        pb."Flera kommuner",
        pb."YH-poäng",
        pb."Studieform",
        pb."Studietakt %" as "Studietakt %",
        pb."Utbildningsanordnare",
        pb."Huvudmannatyp",
        pb."Sökta utbildningsomgångar",
        pb."Beviljade utbildningsomgångar",
        pb."Sökta platser totalt",
        pb."Beviljade platser totalt",
        pb."Sökta platser per utbildningsomgång"
    from rel_program_beslut pb
    join rel_diarie_kommun dk
        on dk.Diarienummer = pb.Diarienummer
    where
        pb.Län = 'Flera kommuner' and pb.Kommun = 'Flera kommuner'
    """
)

rel_flera_kommuner

┌──────────────────┬───────────────┬─────────┬─────────────────────────────────────────┬────────────────────────────────────────────────────────┬─────────────────┬──────────────┬────────────────┬────────────────┬──────────┬────────────┬──────────────┬───────────────────────────────────────────────────────────────────────────┬───────────────┬───────────────────────────┬───────────────────────────────┬──────────────────────┬──────────────────────────┬─────────────────────────────────────┐
│ Ansökningsomgång │ Diarienummer  │ Beslut  │            Utbildningsområde            │                    Utbildningsnamn                     │       Län       │    Kommun    │ Antal kommuner │ Flera kommuner │ YH-poäng │ Studieform │ Studietakt % │                           Utbildningsanordnare                            │ Huvudmannatyp │ Sökta utbildningsomgångar │ Beviljade utbildningsomgångar │ Sökta platser totalt │ Beviljade platser totalt │ Sökta platser per utbildningsomgång │
│      int64    

In [305]:
rel_singel_kommuner = ddb.sql(
    """
    select *
    from rel_program_beslut pb
    where Län != 'Flera kommuner' and Kommun != 'Flera kommuner'
    """
)

rel_singel_kommuner

┌───────────────────┬────────────────────────────────────────────────────────────┬─────────────────┬─────────────┬────────────────┬────────────────┬──────────┬────────────┬──────────────┬────────────────────────────────────┬───────────────┬───────────────────────────┬───────────────────────────────┬──────────────────────┬──────────────────────────┬─────────────────────────────────────┬──────────────────┬───────────────┬─────────┐
│ Utbildningsområde │                      Utbildningsnamn                       │       Län       │   Kommun    │ Antal kommuner │ Flera kommuner │ YH-poäng │ Studieform │ Studietakt % │        Utbildningsanordnare        │ Huvudmannatyp │ Sökta utbildningsomgångar │ Beviljade utbildningsomgångar │ Sökta platser totalt │ Beviljade platser totalt │ Sökta platser per utbildningsomgång │ Ansökningsomgång │ Diarienummer  │ Beslut  │
│      varchar      │                          varchar                           │     varchar     │   varchar   │     int64      │ 

In [306]:
rel_kommuner = ddb.sql(
    """
    select
        Diarienummer,
        Beslut,
        "Flera kommuner",
        "Antal kommuner",
        Län,
        Kommun,
    from rel_program_beslut
    where
        "Flera kommuner" = false
        or "Antal kommuner" = 1
    """
)

In [307]:
#         -- ("Flera kommuner" = true or "Antal kommuner" > 1)

In [314]:
rel_alla_kommuner = ddb.sql(
    """
    select
        pb."Ansökningsomgång",
        pb."Diarienummer",
        pb."Beslut",
        pb."Utbildningsområde",
        pb."Utbildningsnamn",
        pb."Län",
        pb."Kommun",
        pb."Antal kommuner",
        pb."Flera kommuner",
        pb."YH-poäng",
        pb."Studieform",
        pb."Studietakt %" as "Studietakt %",
        pb."Utbildningsanordnare",
        pb."Huvudmannatyp",
        pb."Sökta utbildningsomgångar",
        pb."Beviljade utbildningsomgångar",
        pb."Sökta platser totalt",
        pb."Beviljade platser totalt",
        pb."Sökta platser per utbildningsomgång"
    from rel_program_beslut pb
    where Län != 'Flera kommuner' and Kommun != 'Flera kommuner'

    union all

    select
        pb."Ansökningsomgång",
        pb."Diarienummer",
        pb."Beslut",
        pb."Utbildningsområde",
        pb."Utbildningsnamn",
        dk."Län",                -- 'Flera kommuner'
        dk."Kommun",             -- 'Flera kommuner'
        pb."Antal kommuner",
        pb."Flera kommuner",
        pb."YH-poäng",
        pb."Studieform",
        pb."Studietakt %" as "Studietakt %",
        pb."Utbildningsanordnare",
        pb."Huvudmannatyp",
        pb."Sökta utbildningsomgångar",
        pb."Beviljade utbildningsomgångar",
        pb."Sökta platser totalt",
        pb."Beviljade platser totalt",
        pb."Sökta platser per utbildningsomgång"
    from rel_program_beslut pb
    join rel_diarie_kommun dk
        on dk.Diarienummer = pb.Diarienummer
    where
        pb.Län = 'Flera kommuner' and pb.Kommun = 'Flera kommuner'
    """
)

rel_alla_kommuner

# rel_alla_kommuner.to_parquet("data/resultat-ansokningsomgang-2020-2024-alla_kommuner.parquet")

┌──────────────────┬───────────────┬─────────┬─────────────────────────────────────────┬────────────────────────────────────────────────────────┬─────────────────┬──────────────┬────────────────┬────────────────┬──────────┬────────────┬──────────────┬───────────────────────────────────────────────────────────────────────────┬───────────────┬───────────────────────────┬───────────────────────────────┬──────────────────────┬──────────────────────────┬─────────────────────────────────────┐
│ Ansökningsomgång │ Diarienummer  │ Beslut  │            Utbildningsområde            │                    Utbildningsnamn                     │       Län       │    Kommun    │ Antal kommuner │ Flera kommuner │ YH-poäng │ Studieform │ Studietakt % │                           Utbildningsanordnare                            │ Huvudmannatyp │ Sökta utbildningsomgångar │ Beviljade utbildningsomgångar │ Sökta platser totalt │ Beviljade platser totalt │ Sökta platser per utbildningsomgång │
│      int64    

In [309]:
ddb.sql(
    """
SELECT 
  SUM(Antal_kommuner) AS total_antal_kommuner
FROM (
  SELECT 
    "Antal kommuner" AS Antal_kommuner,
    ROW_NUMBER() OVER (PARTITION BY "Diarienummer" ORDER BY (SELECT 1)) AS rn
  FROM rel_alla_kommuner
) subquery
WHERE rn = 1;
    """
)

┌──────────────────────┐
│ total_antal_kommuner │
│        int128        │
├──────────────────────┤
│                 7526 │
└──────────────────────┘

In [310]:
rel_alla_kommuner = ddb.read_parquet("data/resultat-ansokningsomgang-2020-2024-alla_kommuner.parquet")
rel_alla_kommuner

┌──────────────────┬───────────────┬─────────┬─────────────────────────────────────────┬────────────────────────────────────────────────────────┬─────────────────┬──────────────┬────────────────┬────────────────┬──────────┬────────────┬──────────────┬───────────────────────────────────────────────────────────────────────────┬───────────────┬───────────────────────────┬───────────────────────────────┬──────────────────────┬──────────────────────────┬─────────────────────────────────────┐
│ Ansökningsomgång │ Diarienummer  │ Beslut  │            Utbildningsområde            │                    Utbildningsnamn                     │       Län       │    Kommun    │ Antal kommuner │ Flera kommuner │ YH-poäng │ Studieform │ Studietakt % │                           Utbildningsanordnare                            │ Huvudmannatyp │ Sökta utbildningsomgångar │ Beviljade utbildningsomgångar │ Sökta platser totalt │ Beviljade platser totalt │ Sökta platser per utbildningsomgång │
│      int64    

In [311]:
rel_alla_kommuner = ddb.sql(
    """
    select
        pb."Ansökningsomgång",
        pb."Diarienummer",
        pb."Beslut",
        pb."Utbildningsområde",
        pb."Utbildningsnamn",
        case
            when pb."Län" = 'Flera kommuner'
            then coalesce(dk."Län", pb."Län")
        end as "Län",
        case
            when pb."Kommun" = 'Flera kommuner'
            then coalesce(dk."Kommun", pb."Kommun")
        end as "Kommun",
        pb."Antal kommuner",
        pb."Flera kommuner",
        pb."YH-poäng",
        pb."Studieform",
        pb."Studietakt %" as "Studietakt %",
        pb."Utbildningsanordnare",
        pb."Huvudmannatyp",
        pb."Sökta utbildningsomgångar",
        pb."Beviljade utbildningsomgångar",
        pb."Sökta platser totalt",
        pb."Beviljade platser totalt",
        pb."Sökta platser per utbildningsomgång"
    from rel_program_beslut pb
    left join rel_diarie_kommun dk
        on dk.Diarienummer = pb.Diarienummer
        and ( pb.Län = 'Flera kommuner' )  -- and pb.Kommun = 'Flera kommuner'
    """
)

rel_alla_kommuner

┌──────────────────┬───────────────┬─────────┬───────────────────┬────────────────────────────────────────────────────────────┬────────────────┬────────────────┬────────────────┬────────────────┬──────────┬────────────┬──────────────┬───────────────────────────────────────────────────────────────────────────┬───────────────┬───────────────────────────┬───────────────────────────────┬──────────────────────┬──────────────────────────┬─────────────────────────────────────┐
│ Ansökningsomgång │ Diarienummer  │ Beslut  │ Utbildningsområde │                      Utbildningsnamn                       │      Län       │     Kommun     │ Antal kommuner │ Flera kommuner │ YH-poäng │ Studieform │ Studietakt % │                           Utbildningsanordnare                            │ Huvudmannatyp │ Sökta utbildningsomgångar │ Beviljade utbildningsomgångar │ Sökta platser totalt │ Beviljade platser totalt │ Sökta platser per utbildningsomgång │
│      int64       │    varchar    │ boolean │    

In [312]:
ddb.sql(
    """
SELECT 
  SUM(Antal_kommuner) AS total_antal_kommuner
FROM (
  SELECT 
    "Antal kommuner" AS Antal_kommuner,
    ROW_NUMBER() OVER (PARTITION BY "Diarienummer" ORDER BY (SELECT 1)) AS rn
  FROM rel_alla_kommuner
) subquery
WHERE rn = 1;
    """
)

┌──────────────────────┐
│ total_antal_kommuner │
│        int128        │
├──────────────────────┤
│                 8589 │
└──────────────────────┘

In [316]:
ddb.sql(
    """--
    SELECT *
FROM rel_program_beslut pb
WHERE pb.Län = 'Flera kommuner'
  AND pb.Kommun = 'Flera kommuner'
  AND pb.Diarienummer NOT IN (SELECT Diarienummer FROM rel_diarie_kommun);

    """
)

┌─────────────────────────┬──────────────────────────────────────────────────┬────────────────┬────────────────┬────────────────┬────────────────┬──────────┬────────────┬──────────────┬───────────────────────────────────────────────────────────────────────────┬───────────────┬───────────────────────────┬───────────────────────────────┬──────────────────────┬──────────────────────────┬─────────────────────────────────────┬──────────────────┬───────────────┬─────────┐
│    Utbildningsområde    │                 Utbildningsnamn                  │      Län       │     Kommun     │ Antal kommuner │ Flera kommuner │ YH-poäng │ Studieform │ Studietakt % │                           Utbildningsanordnare                            │ Huvudmannatyp │ Sökta utbildningsomgångar │ Beviljade utbildningsomgångar │ Sökta platser totalt │ Beviljade platser totalt │ Sökta platser per utbildningsomgång │ Ansökningsomgång │ Diarienummer  │ Beslut  │
│         varchar         │                     varchar   